In [1]:
# # Real-World Data Analysis: *Phyllanthus Niruri* and Kidney Stone Treatments
# 
# This notebook implements a streamlined analysis of kidney stone treatment effectiveness and adverse effects across three platforms: WebMD, Amazon, and Reddit.

# %%
# Import necessary libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import fisher_exact
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
from pathlib import Path

In [ ]:
# ## Helper Functions
# 
# These functions handle data loading, preprocessing, and analysis.

# %%
# -------------------------------------------------
# Configuration and helper functions
# -------------------------------------------------

# ---------- CONFIG ----------
CSV_DIR = Path("csv-files")          # adapt if your CSVs live elsewhere
FILE_MAP = {
    "WebMD":  CSV_DIR / "Kidney Stone Reviews - Reviews - WebMD.csv",
    "Amazon": CSV_DIR / "Kidney Stone Reviews - Reviews - Amazon.csv",
    "Reddit": CSV_DIR / "Kidney Stone Reviews - Reviews - Reddit.csv",
}
HELP_COL = "Helps overall with kidney stones"
AE_COL   = "Side effects mentioned"
HQ_COL   = "Super high quality"
# Statistical decision thresholds
MIN_SAMPLE_SIZE = 30        # Minimum sample size for logistic regression
MIN_EVENT_COUNT = 10        # Minimum number of events for logistic regression
# -----------------------------

def _clean_help_col(series: pd.Series) -> pd.Series:
    """
    Normalise HELP_COL so we can separate 'no information' from explicit 'no'.
    Assumes:  1 = helped, 0 or NaN = no info, −1 = explicitly said it did NOT help.
    """
    mapping = {True: 1, False: -1}          # in case booleans slipped in
    return (series.replace(mapping)
                  .where(series.isin([1, -1]))    # leave 1 / -1 unchanged
                  .astype("Int64"))                # keep NA as <NA>

def load_kidney_stone_data() -> dict[str, pd.DataFrame]:
    """Read the three CSVs into a dict of DataFrames and make sure key columns are typed."""
    data = {}
    for platform, path in FILE_MAP.items():
        df = pd.read_csv(path)
        # Normalise the two key columns
        df[HELP_COL] = _clean_help_col(df[HELP_COL])
        # AE column is 1 if any text is present, else 0
        df[AE_COL] = df[AE_COL].apply(lambda x: 1 if isinstance(x, str) and x.strip() else 0)
        
        # Add Weight column (1.0 by default)
        df['Weight'] = 1.0
        
        # Add extra weight for Chanca piedra in Amazon
        if platform == "Amazon":
            chanca_piedra_corrections = {
                "NaturalisimoLife Chanca Piedra 1600 mg": {
                    "scraped": [82, 18, 37, 103, 146],  # 1-5 stars
                    "actual": [82, 18, 38, 103, 906]
                },
                "EU Natural: \"Stone Breaker\" chanca piedra": {
                    "scraped": [102, 44, 50, 100, 138],  # 1-5 stars
                    "actual": [123, 44, 50, 131, 918]
                }
            }
            
            for brand, correction in chanca_piedra_corrections.items():
                for i in range(5):
                    star = i + 1
                    scraped = correction['scraped'][i]
                    actual = correction['actual'][i]
                    if scraped > 0:
                        weight = actual / scraped
                        mask = (df['Medicine'] == 'Chanca piedra') & (df['Source'] == brand) & (df['Stars'] == star)
                        df.loc[mask, 'Weight'] = weight
        
        data[platform] = df
    return data
    
def prepare_subset(df, *, keep_products=None, min_n=MIN_SAMPLE_SIZE, hq_only=False):
    """
    Returns df filtered by product list and/or high‑quality flag,
    and drops products with N < min_n.
    """
    if hq_only and HQ_COL in df.columns:
        df = df[df[HQ_COL] == 1]

    if keep_products is not None:
        df = df[df["Medicine"].isin(keep_products)]

    # Drop medicines with < min_n rows
    counts = df["Medicine"].value_counts()
    df = df[df["Medicine"].isin(counts[counts >= min_n].index)]

    return df.copy()

def add_wilson(df, num_col, denom_col, prefix):
    """Append % and Wilson CI columns to df and return it."""
    low, high = proportion_confint(df[num_col], df[denom_col], method="wilson")
    df[f"{prefix} %"]       = 100 * df[num_col] / df[denom_col]
    df[f"{prefix} CI low"]  = 100 * low
    df[f"{prefix} CI high"] = 100 * high
    return df

def should_use_fisher(df, product, column, min_n=MIN_SAMPLE_SIZE, min_events=MIN_EVENT_COUNT):
    """
    Determine if Fisher's exact test should be used instead of logistic regression.
    Returns True if Fisher's test is recommended (N < min_n or events < min_events)
    """
    total_n = df[df["Medicine"] == product].shape[0]
    
    if column == HELP_COL:
        events = df[(df["Medicine"] == product) & (df[HELP_COL] == 1)].shape[0]
    else:  # AE_COL
        events = df[(df["Medicine"] == product) & (df[AE_COL] == 1)].shape[0]
    
    return total_n < min_n or events < min_events

def odds_ratio_table(model):
    """Extract odds ratios and CIs from a fitted statsmodels model."""
    or_tab = pd.DataFrame({
        "OR": np.exp(model.params),
        "CI low": np.exp(model.conf_int()[0]),
        "CI high": np.exp(model.conf_int()[1]),
        "p": model.pvalues
    })
    # drop intercept, round nicely
    return (or_tab.loc[~or_tab.index.str.contains("Intercept")]
                   .round({"OR": 2, "CI low": 2, "CI high": 2, "p": 3}))

def fisher_test_ratio(df, ref_med, comp_med, outcome_col):
    """
    Run Fisher's exact test comparing two medicines using weights from the dataframe.
    Returns odds ratio as (comp_med_success/comp_med_failure) / (ref_med_success/ref_med_failure)
    to match logistic regression interpretation where ref_med is the reference level.
    """
    df = df.copy()
    
    # Create binary outcome
    df['binary_outcome'] = df[outcome_col].eq(1).fillna(False).astype(int)
    
    # Calculate weighted counts for 2x2 table
    ref_df = df[df['Medicine'] == ref_med]
    comp_df = df[df['Medicine'] == comp_med]
    
    ref_pos = ref_df[ref_df['binary_outcome'] == 1]['Weight'].sum() if len(ref_df) > 0 else 0
    ref_neg = ref_df[ref_df['binary_outcome'] == 0]['Weight'].sum() if len(ref_df) > 0 else 0
    comp_pos = comp_df[comp_df['binary_outcome'] == 1]['Weight'].sum() if len(comp_df) > 0 else 0
    comp_neg = comp_df[comp_df['binary_outcome'] == 0]['Weight'].sum() if len(comp_df) > 0 else 0
    
    # Round for Fisher's exact test
    a = round(ref_pos)
    b = round(ref_neg)
    c = round(comp_pos)
    d = round(comp_neg)
    
    # Apply Haldane correction if any cell is zero
    if a == 0 or b == 0 or c == 0 or d == 0:
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    
    # Calculate odds ratio directly as (comp_odds / ref_odds) to match regression
    # This is (c/d) / (a/b) = (c*b) / (a*d)
    odds_ratio = (c * b) / (a * d)
    
    # Calculate log odds ratio and standard error for CI
    import math
    log_or = math.log(odds_ratio)
    se_log_or = math.sqrt(1/a + 1/b + 1/c + 1/d)
    
    # Calculate confidence interval
    ci_low = math.exp(log_or - 1.96 * se_log_or)
    ci_high = math.exp(log_or + 1.96 * se_log_or)
    
    # Calculate p-value using Fisher's exact test
    # Note: We use table orientation [[a,b],[c,d]] for the p-value calculation
    _, pvalue = fisher_exact([[a, b], [c, d]])
    
    return odds_ratio, pvalue, (ci_low, ci_high)

In [9]:
# ## 1. Consolidated Summary Table
# 
# We'll create a comprehensive table showing effectiveness and adverse events across all platforms
# for both all reviews and high-quality reviews.

# %%
def create_effectiveness_table():
    """
    Create a table focused on effectiveness metrics across platforms:
    - Platform (WebMD, Amazon, Reddit)
    - Medicine name
    - Sample sizes (All / HQ)
    - Effectiveness percentages (All / HQ)
    """
    data = load_kidney_stone_data()
    
    # Products to exclude from effectiveness table (only for WebMD platform)
    webmd_excluded_products = ["Garcinia", "Black seed", "Ashwagandha", "Melatonin"]
    
    # Prepare an empty list to store effectiveness records
    effectiveness_summaries = []
    
    # Process each platform
    for platform, df in data.items():
        # Get all products with at least 5 reviews
        all_products = df["Medicine"].value_counts()[df["Medicine"].value_counts() >= 5].index.tolist()
        
        # Apply exclusion only for WebMD platform
        if platform == "WebMD":
            all_products = [product for product in all_products if product not in webmd_excluded_products]
        
        for product in all_products:
            # ALL reviews
            all_df = df[df["Medicine"] == product]
            n_all = len(all_df)
            
            # Initialize variables
            helped_pct_all = 0
            helped_ci_low_all = float('nan')
            helped_ci_high_all = float('nan')
            n_hq = 0
            helped_pct_hq = float('nan')
            helped_ci_low_hq = float('nan')
            helped_ci_high_hq = float('nan')
            
            # Calculate effectiveness using weights
            helped_mask = all_df[HELP_COL].eq(1).fillna(False)
            weighted_helped = all_df.loc[helped_mask, 'Weight'].sum() if any(helped_mask) else 0
            total_weight = all_df['Weight'].sum()
            helped_pct_all = 100 * weighted_helped / total_weight if total_weight > 0 else 0
            
            # Add Wilson CIs
            if n_all > 0:
                # Effective counts for CI calculation
                effective_count = weighted_helped / total_weight * n_all if total_weight > 0 else 0
                if effective_count > 0:
                    helped_ci_low_all, helped_ci_high_all = proportion_confint(
                        effective_count, n_all, method="wilson"
                    )
                    helped_ci_low_all *= 100
                    helped_ci_high_all *= 100
            
            # HIGH QUALITY reviews (if applicable)
            if HQ_COL in df.columns:
                hq_df = all_df[all_df[HQ_COL] == 1]
                n_hq = len(hq_df)
                
                if n_hq > 0:
                    # Calculate effectiveness for HQ
                    helped_mask_hq = hq_df[HELP_COL].eq(1).fillna(False)
                    helped_weighted_hq = hq_df.loc[helped_mask_hq, 'Weight'].sum() if any(helped_mask_hq) else 0
                    total_weight_hq = hq_df['Weight'].sum()
                    helped_pct_hq = 100 * helped_weighted_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # For CI calculation, use effective counts
                    if n_hq > 0:
                        effective_helped_hq = helped_weighted_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_helped_hq > 0:
                            helped_ci_low_hq, helped_ci_high_hq = proportion_confint(
                                effective_helped_hq, n_hq, method="wilson"
                            )
                            helped_ci_low_hq *= 100
                            helped_ci_high_hq *= 100
            
            # Add record to summaries
            effectiveness_summaries.append({
                "Platform": platform,
                "Medicine": product,
                "N_All": n_all,
                "N_HQ": n_hq,
                "Helped_Pct_All": helped_pct_all,
                "Helped_CI_Low_All": helped_ci_low_all,
                "Helped_CI_High_All": helped_ci_high_all,
                "Helped_Pct_HQ": helped_pct_hq,
                "Helped_CI_Low_HQ": helped_ci_low_hq,
                "Helped_CI_High_HQ": helped_ci_high_hq
            })
    
    # Convert to DataFrame and format
    effectiveness_df = pd.DataFrame(effectiveness_summaries)
    
    # Create a more readable format for effectiveness with CIs
    effectiveness_df["Effectiveness_All"] = effectiveness_df.apply(
        lambda x: f"{x['Helped_Pct_All']:.1f}% ({x['Helped_CI_Low_All']:.1f}-{x['Helped_CI_High_All']:.1f})" 
        if not pd.isna(x['Helped_CI_Low_All']) else f"{x['Helped_Pct_All']:.1f}%", 
        axis=1
    )
    
    effectiveness_df["Effectiveness_HQ"] = effectiveness_df.apply(
        lambda x: f"{x['Helped_Pct_HQ']:.1f}% ({x['Helped_CI_Low_HQ']:.1f}-{x['Helped_CI_High_HQ']:.1f})" 
        if not pd.isna(x['Helped_CI_Low_HQ']) else f"{x['Helped_Pct_HQ']:.1f}%" if not pd.isna(x['Helped_Pct_HQ']) else "N/A", 
        axis=1
    )
    
    # Select and order columns for final display
    display_cols = [
        "Platform", "Medicine", 
        "N_All", "Effectiveness_All",
        "N_HQ", "Effectiveness_HQ"
    ]
    
    # Sort by platform and then by medicine
    effectiveness_df = effectiveness_df.sort_values(["Platform", "Medicine"])
    
    return effectiveness_df[display_cols]

def create_adverse_events_table():
    """
    Create a table focused on adverse events metrics across platforms:
    - Platform (WebMD, Amazon, Reddit)
    - Medicine name
    - Sample sizes (All / HQ)
    - Adverse Events percentages (All / HQ)
    """
    data = load_kidney_stone_data()
    
    # Prepare an empty list to store adverse events records
    ae_summaries = []
    
    # Process each platform
    for platform, df in data.items():
        # Get all products with at least 5 reviews
        all_products = df["Medicine"].value_counts()[df["Medicine"].value_counts() >= 5].index.tolist()
        
        for product in all_products:
            # ALL reviews
            all_df = df[df["Medicine"] == product]
            n_all = len(all_df)
            
            # Initialize variables
            ae_pct_all = 0
            ae_ci_low_all = float('nan')
            ae_ci_high_all = float('nan')
            n_hq = 0
            ae_pct_hq = float('nan')
            ae_ci_low_hq = float('nan')
            ae_ci_high_hq = float('nan')
            
            # Calculate adverse events
            ae_mask = all_df[AE_COL].eq(1).fillna(False)
            ae_weighted = all_df.loc[ae_mask, 'Weight'].sum() if any(ae_mask) else 0
            total_weight = all_df['Weight'].sum()
            
            # Calculate percentage using weights
            ae_pct_all = 100 * ae_weighted / total_weight if total_weight > 0 else 0
            
            # For CI calculation, use effective counts
            if n_all > 0:
                effective_ae = ae_weighted / total_weight * n_all if total_weight > 0 else 0
                if effective_ae > 0:
                    ae_ci_low_all, ae_ci_high_all = proportion_confint(
                        effective_ae, n_all, method="wilson"
                    )
                    ae_ci_low_all *= 100
                    ae_ci_high_all *= 100
            
            # HIGH QUALITY reviews (if applicable)
            if HQ_COL in df.columns:
                hq_df = all_df[all_df[HQ_COL] == 1]
                n_hq = len(hq_df)
                
                if n_hq > 0:
                    # Calculate adverse events for HQ
                    ae_mask_hq = hq_df[AE_COL].eq(1).fillna(False)
                    ae_weighted_hq = hq_df.loc[ae_mask_hq, 'Weight'].sum() if any(ae_mask_hq) else 0
                    total_weight_hq = hq_df['Weight'].sum()
                    ae_pct_hq = 100 * ae_weighted_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # For CI calculation, use effective counts
                    if n_hq > 0:
                        effective_ae_hq = ae_weighted_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_ae_hq > 0:
                            ae_ci_low_hq, ae_ci_high_hq = proportion_confint(
                                effective_ae_hq, n_hq, method="wilson"
                            )
                            ae_ci_low_hq *= 100
                            ae_ci_high_hq *= 100
            
            # Add record to summaries
            ae_summaries.append({
                "Platform": platform,
                "Medicine": product,
                "N_All": n_all,
                "N_HQ": n_hq,
                "AE_Pct_All": ae_pct_all,
                "AE_CI_Low_All": ae_ci_low_all,
                "AE_CI_High_All": ae_ci_high_all,
                "AE_Pct_HQ": ae_pct_hq,
                "AE_CI_Low_HQ": ae_ci_low_hq,
                "AE_CI_High_HQ": ae_ci_high_hq
            })
    
    # Convert to DataFrame and format
    ae_df = pd.DataFrame(ae_summaries)
    
    # Create a more readable format for AE with CIs
    ae_df["Adverse_Events_All"] = ae_df.apply(
        lambda x: f"{x['AE_Pct_All']:.1f}% ({x['AE_CI_Low_All']:.1f}-{x['AE_CI_High_All']:.1f})" 
        if not pd.isna(x['AE_CI_Low_All']) else f"{x['AE_Pct_All']:.1f}%", 
        axis=1
    )
    
    ae_df["Adverse_Events_HQ"] = ae_df.apply(
        lambda x: f"{x['AE_Pct_HQ']:.1f}% ({x['AE_CI_Low_HQ']:.1f}-{x['AE_CI_High_HQ']:.1f})" 
        if not pd.isna(x['AE_CI_Low_HQ']) else f"{x['AE_Pct_HQ']:.1f}%" if not pd.isna(x['AE_Pct_HQ']) else "N/A", 
        axis=1
    )
    
    # Select and order columns for final display
    display_cols = [
        "Platform", "Medicine", 
        "N_All", "Adverse_Events_All",
        "N_HQ", "Adverse_Events_HQ"
    ]
    
    # Sort by platform and then by medicine
    ae_df = ae_df.sort_values(["Platform", "Medicine"])
    
    return ae_df[display_cols]

# Function to run both tables and save them
def create_and_save_split_tables():
    """
    Create both effectiveness and adverse events tables and save them to CSV
    """
    # Generate the tables
    effectiveness_table = create_effectiveness_table()
    adverse_events_table = create_adverse_events_table()
    
    # Display the tables
    print("Effectiveness Table:")
    display(effectiveness_table)
    
    print("\nAdverse Events Table:")
    display(adverse_events_table)
    
    # Save to CSV for reference
    effectiveness_table.to_csv(CSV_DIR / "effectiveness_summary_table.csv", index=False)
    adverse_events_table.to_csv(CSV_DIR / "adverse_events_summary_table.csv", index=False)
    print("✓ Saved split summary tables to CSV")
    
    return effectiveness_table, adverse_events_table

# Run the function
effectiveness_table, adverse_events_table = create_and_save_split_tables()

Effectiveness Table:


,Platform,Medicine,N_All,Effectiveness_All,N_HQ,Effectiveness_HQ
5,Amazon,Chanca piedra,1193,72.4% (69.8-74.9),158,87.4% (81.4-91.7)
8,Amazon,Phosfood,40,75.0% (59.8-85.8),1,100.0% (20.7-100.0)
6,Amazon,Potassium citrate,133,91.0% (84.9-94.8),26,92.3% (75.9-97.9)
7,Amazon,Rowatinex,90,91.1% (83.4-95.4),20,100.0% (83.9-100.0)
12,Reddit,Allopurinol,49,20.4% (11.5-33.6),8,50.0% (21.5-78.5)
16,Reddit,Black seed,17,23.5% (9.6-47.3),3,33.3% (6.1-79.2)
11,Reddit,Chanca piedra,492,35.2% (31.1-39.5),75,61.3% (50.0-71.5)
9,Reddit,Flomax,1134,24.6% (22.2-27.2),232,40.5% (34.4-46.9)
14,Reddit,Garcinia,38,42.1% (27.9-57.8),1,0.0%
13,Reddit,Hydrochlorothiazide,46,32.6% (20.9-47.0),6,33.3% (9.7-70.0)



Adverse Events Table:


,Platform,Medicine,N_All,Adverse_Events_All,N_HQ,Adverse_Events_HQ
9,Amazon,Chanca piedra,1193,2.7% (1.9-3.7),158,1.6% (0.5-5.0)
12,Amazon,Phosfood,40,7.5% (2.6-19.9),1,0.0%
10,Amazon,Potassium citrate,133,2.3% (0.8-6.4),26,3.8% (0.7-18.9)
11,Amazon,Rowatinex,90,2.2% (0.6-7.7),20,10.0% (2.8-30.1)
16,Reddit,Allopurinol,49,12.2% (5.7-24.2),8,12.5% (2.2-47.1)
20,Reddit,Black seed,17,0.0%,3,0.0%
15,Reddit,Chanca piedra,492,6.5% (4.6-9.0),75,14.7% (8.4-24.4)
13,Reddit,Flomax,1134,21.9% (19.6-24.4),232,25.0% (19.9-30.9)
18,Reddit,Garcinia,38,13.2% (5.8-27.3),1,100.0% (20.7-100.0)
17,Reddit,Hydrochlorothiazide,46,8.7% (3.4-20.3),6,16.7% (3.0-56.4)


✓ Saved split summary tables to CSV


In [4]:
# ## 2. Multi-Panel Forest Plots
# 
# Two visualizations:
# 1. A multi-panel forest plot showing effectiveness across platforms
# 2. A multi-panel forest plot showing adverse events across platforms
def create_multipanel_forest_plots(min_n=1):
    data = load_kidney_stone_data()
    platforms = list(data.keys())
    
    # Prepare data for plotting
    effectiveness_data = []
    adverse_events_data = []
    exclude_eff = {"Melatonin", "Ashwagandha"}

    for platform, df in data.items():
        products = df["Medicine"].value_counts()[df["Medicine"].value_counts() >= min_n].index.tolist()
        
        for product in products:
            # ALL reviews
            all_df = df[df["Medicine"] == product]
            n_all = len(all_df)
            
            # Calculate effectiveness using weights
            helped_mask = all_df[HELP_COL].eq(1).fillna(False)
            weighted_helped = all_df.loc[helped_mask, 'Weight'].sum()
            total_weight = all_df['Weight'].sum()
            helped_pct_all = 100 * weighted_helped / total_weight if total_weight > 0 else 0
            
            # Calculate CI
            if n_all > 0:
                effective_count = weighted_helped / total_weight * n_all if total_weight > 0 else 0
                if effective_count > 0:
                    helped_ci_low_all, helped_ci_high_all = proportion_confint(
                        effective_count, n_all, method="wilson"
                    )
                    helped_ci_low_all *= 100
                    helped_ci_high_all *= 100
                else:
                    helped_ci_low_all = helped_ci_high_all = np.nan
            else:
                helped_ci_low_all = helped_ci_high_all = np.nan
            
            # Calculate adverse events using weights
            ae_mask = all_df[AE_COL].eq(1).fillna(False)
            weighted_ae = all_df.loc[ae_mask, 'Weight'].sum()
            ae_pct_all = 100 * weighted_ae / total_weight if total_weight > 0 else 0
            
            # Calculate CI
            if n_all > 0:
                effective_count = weighted_ae / total_weight * n_all if total_weight > 0 else 0
                if effective_count > 0:
                    ae_ci_low_all, ae_ci_high_all = proportion_confint(
                        effective_count, n_all, method="wilson"
                    )
                    ae_ci_low_all *= 100
                    ae_ci_high_all *= 100
                else:
                    ae_ci_low_all = ae_ci_high_all = np.nan
            else:
                ae_ci_low_all = ae_ci_high_all = np.nan
            
            # Add to data
            if product not in exclude_eff:
                effectiveness_data.append({
                    "Platform": platform,
                    "Medicine": product,
                    "Type": "All Reviews",
                    "Percentage": helped_pct_all,
                    "CI_Low": helped_ci_low_all,
                    "CI_High": helped_ci_high_all,
                    "N": n_all
                })
            
            adverse_events_data.append({
                "Platform": platform,
                "Medicine": product,
                "Type": "All Reviews",
                "Percentage": ae_pct_all,
                "CI_Low": ae_ci_low_all,
                "CI_High": ae_ci_high_all,
                "N": n_all
            })
            
            # HIGH QUALITY reviews
            if HQ_COL in df.columns:
                hq_df = all_df[all_df[HQ_COL] == 1]
                n_hq = len(hq_df)
                
                if n_hq >= min_n:
                    # Calculate effectiveness using weights
                    helped_mask_hq = hq_df[HELP_COL].eq(1).fillna(False)
                    weighted_helped_hq = hq_df.loc[helped_mask_hq, 'Weight'].sum()
                    total_weight_hq = hq_df['Weight'].sum()
                    helped_pct_hq = 100 * weighted_helped_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # Calculate CI
                    if n_hq > 0:
                        effective_count = weighted_helped_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_count > 0:
                            helped_ci_low_hq, helped_ci_high_hq = proportion_confint(
                                effective_count, n_hq, method="wilson"
                            )
                            helped_ci_low_hq *= 100
                            helped_ci_high_hq *= 100
                        else:
                            helped_ci_low_hq = helped_ci_high_hq = np.nan
                    else:
                        helped_ci_low_hq = helped_ci_high_hq = np.nan
                    
                    # Calculate adverse events using weights
                    ae_mask_hq = hq_df[AE_COL].eq(1).fillna(False)
                    weighted_ae_hq = hq_df.loc[ae_mask_hq, 'Weight'].sum()
                    ae_pct_hq = 100 * weighted_ae_hq / total_weight_hq if total_weight_hq > 0 else 0
                    
                    # Calculate CI
                    if n_hq > 0:
                        effective_count = weighted_ae_hq / total_weight_hq * n_hq if total_weight_hq > 0 else 0
                        if effective_count > 0:
                            ae_ci_low_hq, ae_ci_high_hq = proportion_confint(
                                effective_count, n_hq, method="wilson"
                            )
                            ae_ci_low_hq *= 100
                            ae_ci_high_hq *= 100
                        else:
                            ae_ci_low_hq = ae_ci_high_hq = np.nan
                    else:
                        ae_ci_low_hq = ae_ci_high_hq = np.nan
                    
                    # Add to data
                    if product not in exclude_eff:
                        effectiveness_data.append({
                            "Platform": platform,
                            "Medicine": product,
                            "Type": "High Quality",
                            "Percentage": helped_pct_hq,
                            "CI_Low": helped_ci_low_hq,
                            "CI_High": helped_ci_high_hq,
                            "N": n_hq
                        })
                    
                    adverse_events_data.append({
                        "Platform": platform,
                        "Medicine": product,
                        "Type": "High Quality",
                        "Percentage": ae_pct_hq,
                        "CI_Low": ae_ci_low_hq,
                        "CI_High": ae_ci_high_hq,
                        "N": n_hq
                    })
    
    # Convert to DataFrames
    effectiveness_df = pd.DataFrame(effectiveness_data)
    adverse_events_df = pd.DataFrame(adverse_events_data)
    
    # Create plots
    create_panel_forest_plot(effectiveness_df, "Effectiveness", "% Reporting Help")
    create_panel_forest_plot(adverse_events_df, "Adverse Events", "% Reporting Adverse Events")

def create_panel_forest_plot(data_df, plot_title, y_axis_title):
    """
    Create a multi-panel forest plot showing results across platforms.
    
    Parameters:
    - data_df: DataFrame with columns Platform, Medicine, Type, Percentage, CI_Low, CI_High, N
    - plot_title: Title for the overall plot
    - y_axis_title: Label for the y-axis
    """
    # Set platform order and colors
    platform_order = ["WebMD", "Amazon", "Reddit"]
    platform_colors = {"WebMD": "#636EFA", "Amazon": "#EF553B", "Reddit": "#00CC96"}
    review_type_colors = {"All Reviews": "solid", "High Quality": "rgba(0,0,0,0.5)"}
    
    # Set marker styles for review types
    marker_styles = {"All Reviews": "circle", "High Quality": "diamond"}
    
    # Create multi-panel figure
    from plotly.subplots import make_subplots
    fig = make_subplots(rows=1, cols=3, 
                        subplot_titles=platform_order,
                        shared_yaxes=True,
                        horizontal_spacing=0.05)  # Increased spacing
    
    # Get the unique medications across all platforms for consistent y-axis
    all_meds = data_df["Medicine"].unique()
    
    # Process each platform
    for i, platform in enumerate(platform_order):
        # Filter data for this platform
        platform_data = data_df[data_df["Platform"] == platform]
        
        if len(platform_data) == 0:
            continue
            
        # Sort by percentage (descending) for All Reviews
        all_reviews_data = platform_data[platform_data["Type"] == "All Reviews"]
        med_order = all_reviews_data.sort_values("Percentage", ascending=False)["Medicine"].unique()
        
        # Add All Reviews trace
        all_data = platform_data[platform_data["Type"] == "All Reviews"]
        
        # Add trace for All Reviews - use medicine names directly
        fig.add_trace(
            go.Scatter(
                x=all_data["Percentage"],
                y=all_data["Medicine"],  # Use medicine names directly
                error_x=dict(
                    type='data',
                    symmetric=False,
                    array=all_data["CI_High"] - all_data["Percentage"],
                    arrayminus=all_data["Percentage"] - all_data["CI_Low"],
                    color=platform_colors[platform],
                ),
                mode="markers",
                marker=dict(
                    symbol="circle",
                    size=10,
                    color=platform_colors[platform],
                    opacity=1.0,
                    line=dict(width=2, color="DarkSlateGrey")
                ),
                name=f"{platform} - All Reviews",
                text=[f"N={n}" for n in all_data["N"]],
                hovertemplate="%{y}: %{x:.1f}% (%{error_x.arrayminus:.1f}-%{error_x.array:.1f})<br>%{text}",
                showlegend=i==0,
            ),
            row=1, col=i+1
        )
        
        # Add High Quality trace if present
        hq_data = platform_data[platform_data["Type"] == "High Quality"]
        if len(hq_data) > 0:
            # Use a simple way to create visual distinction: just use opacity
            fig.add_trace(
                go.Scatter(
                    x=hq_data["Percentage"],
                    y=hq_data["Medicine"],  # Same medicine names as "All Reviews"
                    error_x=dict(
                        type='data',
                        symmetric=False,
                        array=hq_data["CI_High"] - hq_data["Percentage"],
                        arrayminus=hq_data["Percentage"] - hq_data["CI_Low"],
                        color=platform_colors[platform],
                    ),
                    mode="markers",
                    marker=dict(
                        symbol="diamond",  # Different shape
                        size=10,
                        color=platform_colors[platform],
                        opacity=0.6,  # Transparency
                        line=dict(width=2, color="black")
                    ),
                    name=f"{platform} - High Quality",
                    text=[f"N={n}" for n in hq_data["N"]],
                    hovertemplate="%{y}: %{x:.1f}% (%{error_x.arrayminus:.1f}-%{error_x.array:.1f})<br>%{text}",
                    showlegend=i==0,
                ),
                row=1, col=i+1
            )
    
    # Update layout
    fig.update_layout(
        title=plot_title,
        height=max(600, 100 + 40 * len(all_meds)),  # Dynamic height based on number of medicines
        width=1000,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    # Update x-axes
    for i in range(3):
        fig.update_xaxes(
            title=y_axis_title,
            range=[0, 100],
            tickvals=[0, 20, 40, 60, 80, 100],
            ticktext=["0", "20", "40", "60", "80", "100"],
            tickfont=dict(size=10),
            row=1, col=i+1
        )
    
    # Show the plot
    fig.show()
    
    # Save as HTML
    output_path = CSV_DIR / f"forest_plot_{plot_title.lower().replace(' ', '_')}.html"
    # fig.write_html(output_path)
    print(f"✓ Saved forest plot to {output_path}")
    
    # Try to save as PNG if kaleido is installed
    try:
        png_path = CSV_DIR / f"forest_plot_{plot_title.lower().replace(' ', '_')}.png"
        # fig.write_image(png_path, scale=2)
        print(f"✓ Saved PNG version to {png_path}")
    except Exception as e:
        print(f"Note: Could not save PNG (install 'kaleido' package if needed). Error: {e}")
    
    return fig

# Run the function to create the forest plots
multipanel_plots = create_multipanel_forest_plots(min_n=1)

✓ Saved forest plot to csv-files/forest_plot_effectiveness.html
✓ Saved PNG version to csv-files/forest_plot_effectiveness.png


✓ Saved forest plot to csv-files/forest_plot_adverse_events.html
✓ Saved PNG version to csv-files/forest_plot_adverse_events.png


In [5]:
def create_odds_ratio_table(ref_product="Chanca piedra", 
                           min_include_n=5,  # Minimum n to include in table
                           min_regression_n=MIN_SAMPLE_SIZE,  # For Fisher vs regression
                           min_events=MIN_EVENT_COUNT):  # For Fisher vs regression
    """
    Create a table of odds ratios comparing the reference product (default: Chanca piedra)
    to other treatments across platforms.
    
    Parameters:
    -----------
    ref_product : str
        The reference product to compare other treatments against
    min_include_n : int
        Minimum number of reviews required for a product to be included
    min_regression_n : int
        Minimum number of reviews required for regression analysis
    min_events : int
        Minimum number of events required for regression analysis
        
    Returns:
    --------
    pandas.DataFrame
        Table of odds ratios with confidence intervals and p-values
    """
    data = load_kidney_stone_data()
    
    # Prepare an empty list to store all OR records
    all_ors = []
    
    # Process each platform
    for platform, df in data.items():
        # Check if ref_product exists in this platform
        if ref_product not in df["Medicine"].unique():
            continue
        
        # Get products with enough reviews for comparison
        products_with_count = df["Medicine"].value_counts()
        all_products = products_with_count[products_with_count >= min_include_n].index.tolist()
        
        if ref_product not in all_products:
            continue
            
        # For each product (except reference), calculate ORs
        for product in all_products:
            if product == ref_product:
                continue
                
            # ALL reviews
            all_df = prepare_subset(df, keep_products=[ref_product, product], min_n=1)
            n_ref = len(all_df[all_df["Medicine"] == ref_product])
            n_comp = len(all_df[all_df["Medicine"] == product])
            
            # --- EFFECTIVENESS (ALL) ---
            use_fisher_helped = should_use_fisher(all_df, product, HELP_COL,
                                        min_n=min_regression_n,
                                        min_events=min_events) or \
                                should_use_fisher(all_df, ref_product, HELP_COL,
                                        min_n=min_regression_n,
                                        min_events=min_events)
            
            if use_fisher_helped:
                # Use Fisher's exact test for effectiveness (weights handled automatically)
                or_helped, p_helped, (ci_low_helped, ci_high_helped) = fisher_test_ratio(
                    all_df, ref_product, product, HELP_COL
                )
                test_method_helped = "Fisher"
            else:
                # Use logistic regression for effectiveness
                try:
                    # Prepare data for regression
                    reg_df = all_df.copy()
                    reg_df["helped"] = (reg_df[HELP_COL] == 1).fillna(False).astype(int)
                    
                    # Set reference level
                    reg_df["Medicine"] = reg_df["Medicine"].astype("category")
                    reg_df["Medicine"] = reg_df["Medicine"].cat.reorder_categories(
                        [ref_product] + [p for p in reg_df["Medicine"].cat.categories if p != ref_product],
                        ordered=True
                    )
                    
                    # Fit model with weights
                    model_helped = smf.glm(
                        "helped ~ C(Medicine)", 
                        reg_df, 
                        family=sm.families.Binomial(),
                        freq_weights=reg_df['Weight']
                    ).fit()
                    
                    # Extract coefficients
                    param_idx = f"C(Medicine)[T.{product}]"
                    if param_idx in model_helped.params:
                        or_helped = np.exp(model_helped.params[param_idx])
                        ci_low_helped = np.exp(model_helped.conf_int().loc[param_idx, 0])
                        ci_high_helped = np.exp(model_helped.conf_int().loc[param_idx, 1])
                        p_helped = model_helped.pvalues[param_idx]
                    else:
                        # Parameter not in model (rare case)
                        or_helped = ci_low_helped = ci_high_helped = p_helped = None
                    
                    test_method_helped = "Regression"
                except Exception as e:
                    # Fallback to Fisher's test if regression fails
                    or_helped, p_helped, (ci_low_helped, ci_high_helped) = fisher_test_ratio(
                        all_df, ref_product, product, HELP_COL
                    )
                    test_method_helped = "Fisher (fallback)"
            
            # --- ADVERSE EVENTS (ALL) ---
            use_fisher_ae = should_use_fisher(all_df, product, AE_COL,
                                    min_n=min_regression_n,
                                    min_events=min_events) or \
                            should_use_fisher(all_df, ref_product, AE_COL,
                                    min_n=min_regression_n,
                                    min_events=min_events)
            
            if use_fisher_ae:
                # Use Fisher's exact test for adverse events
                or_ae, p_ae, (ci_low_ae, ci_high_ae) = fisher_test_ratio(
                    all_df, ref_product, product, AE_COL
                )
                test_method_ae = "Fisher"
            else:
                # Use logistic regression for adverse events
                try:
                    # Prepare data for regression
                    reg_df = all_df.copy()
                    reg_df["ae"] = reg_df[AE_COL].fillna(0).astype(int)
                    
                    # Set reference level
                    reg_df["Medicine"] = reg_df["Medicine"].astype("category")
                    reg_df["Medicine"] = reg_df["Medicine"].cat.reorder_categories(
                        [ref_product] + [p for p in reg_df["Medicine"].cat.categories if p != ref_product],
                        ordered=True
                    )
                    
                    # Fit model with weights
                    model_ae = smf.glm(
                        "ae ~ C(Medicine)", 
                        reg_df, 
                        family=sm.families.Binomial(),
                        freq_weights=reg_df['Weight']
                    ).fit()
                    
                    # Extract coefficients
                    param_idx = f"C(Medicine)[T.{product}]"
                    if param_idx in model_ae.params:
                        or_ae = np.exp(model_ae.params[param_idx])
                        ci_low_ae = np.exp(model_ae.conf_int().loc[param_idx, 0])
                        ci_high_ae = np.exp(model_ae.conf_int().loc[param_idx, 1])
                        p_ae = model_ae.pvalues[param_idx]
                    else:
                        # Parameter not in model (rare case)
                        or_ae = ci_low_ae = ci_high_ae = p_ae = None
                    
                    test_method_ae = "Regression"
                except Exception as e:
                    # Fallback to Fisher's test if regression fails
                    or_ae, p_ae, (ci_low_ae, ci_high_ae) = fisher_test_ratio(
                        all_df, ref_product, product, AE_COL
                    )
                    test_method_ae = "Fisher (fallback)"
            
            # --- HIGH QUALITY REVIEWS ---
            if HQ_COL in df.columns:
                # Filter to high quality reviews
                hq_df = prepare_subset(df, keep_products=[ref_product, product], min_n=1, hq_only=True)
                n_hq_ref = len(hq_df[hq_df["Medicine"] == ref_product])
                n_hq_comp = len(hq_df[hq_df["Medicine"] == product])
                
                # Only include high quality reviews in the comparison if both products have enough samples
                if n_hq_ref >= min_include_n and n_hq_comp >= min_include_n:
                    # --- EFFECTIVENESS (HQ) ---
                    use_fisher_helped_hq = should_use_fisher(hq_df, product, HELP_COL, 
                                                        min_n=min_regression_n, 
                                                        min_events=min_events) or \
                                            should_use_fisher(hq_df, ref_product, HELP_COL, 
                                                        min_n=min_regression_n, 
                                                        min_events=min_events)
                    
                    if use_fisher_helped_hq:
                        # Use Fisher's exact test for effectiveness
                        or_helped_hq, p_helped_hq, (ci_low_helped_hq, ci_high_helped_hq) = fisher_test_ratio(
                            hq_df, ref_product, product, HELP_COL
                        )
                        test_method_helped_hq = "Fisher"
                    else:
                        # Use logistic regression for effectiveness
                        try:
                            # Prepare data for regression
                            reg_df = hq_df.copy()
                            reg_df["helped"] = (reg_df[HELP_COL] == 1).fillna(False).astype(int)
                            
                            # Set reference level
                            reg_df["Medicine"] = reg_df["Medicine"].astype("category")
                            reg_df["Medicine"] = reg_df["Medicine"].cat.reorder_categories(
                                [ref_product] + [p for p in reg_df["Medicine"].cat.categories if p != ref_product],
                                ordered=True
                            )
                            
                            # Fit model with weights
                            model_helped_hq = smf.glm(
                                "helped ~ C(Medicine)", 
                                reg_df, 
                                family=sm.families.Binomial(),
                                freq_weights=reg_df['Weight']
                            ).fit()
                            
                            # Extract coefficients
                            param_idx = f"C(Medicine)[T.{product}]"
                            if param_idx in model_helped_hq.params:
                                or_helped_hq = np.exp(model_helped_hq.params[param_idx])
                                ci_low_helped_hq = np.exp(model_helped_hq.conf_int().loc[param_idx, 0])
                                ci_high_helped_hq = np.exp(model_helped_hq.conf_int().loc[param_idx, 1])
                                p_helped_hq = model_helped_hq.pvalues[param_idx]
                            else:
                                # Parameter not in model (rare case)
                                or_helped_hq = ci_low_helped_hq = ci_high_helped_hq = p_helped_hq = None
                            
                            test_method_helped_hq = "Regression"
                        except Exception as e:
                            # Fallback to Fisher's test if regression fails
                            or_helped_hq, p_helped_hq, (ci_low_helped_hq, ci_high_helped_hq) = fisher_test_ratio(
                                hq_df, ref_product, product, HELP_COL
                            )
                            test_method_helped_hq = "Fisher (fallback)"
                    
                    # --- ADVERSE EVENTS (HQ) ---
                    use_fisher_ae_hq = should_use_fisher(hq_df, product, AE_COL,
                                            min_n=min_regression_n,
                                            min_events=min_events) or \
                                        should_use_fisher(hq_df, ref_product, AE_COL,
                                            min_n=min_regression_n,
                                            min_events=min_events)
                    
                    if use_fisher_ae_hq:
                        # Use Fisher's exact test for adverse events
                        or_ae_hq, p_ae_hq, (ci_low_ae_hq, ci_high_ae_hq) = fisher_test_ratio(
                            hq_df, ref_product, product, AE_COL
                        )
                        test_method_ae_hq = "Fisher"
                    else:
                        # Use logistic regression for adverse events
                        try:
                            # Prepare data for regression
                            reg_df = hq_df.copy()
                            reg_df["ae"] = reg_df[AE_COL].fillna(0).astype(int)
                            
                            # Set reference level
                            reg_df["Medicine"] = reg_df["Medicine"].astype("category")
                            reg_df["Medicine"] = reg_df["Medicine"].cat.reorder_categories(
                                [ref_product] + [p for p in reg_df["Medicine"].cat.categories if p != ref_product],
                                ordered=True
                            )
                            
                            # Fit model with weights
                            model_ae_hq = smf.glm(
                                "ae ~ C(Medicine)", 
                                reg_df, 
                                family=sm.families.Binomial(),
                                freq_weights=reg_df['Weight']
                            ).fit()
                            
                            # Extract coefficients
                            param_idx = f"C(Medicine)[T.{product}]"
                            if param_idx in model_ae_hq.params:
                                or_ae_hq = np.exp(model_ae_hq.params[param_idx])
                                ci_low_ae_hq = np.exp(model_ae_hq.conf_int().loc[param_idx, 0])
                                ci_high_ae_hq = np.exp(model_ae_hq.conf_int().loc[param_idx, 1])
                                p_ae_hq = model_ae_hq.pvalues[param_idx]
                            else:
                                # Parameter not in model (rare case)
                                or_ae_hq = ci_low_ae_hq = ci_high_ae_hq = p_ae_hq = None
                            
                            test_method_ae_hq = "Regression"
                        except Exception as e:
                            # Fallback to Fisher's test if regression fails
                            or_ae_hq, p_ae_hq, (ci_low_ae_hq, ci_high_ae_hq) = fisher_test_ratio(
                                hq_df, ref_product, product, AE_COL
                            )
                            test_method_ae_hq = "Fisher (fallback)"
                else:
                    # Not enough HQ reviews
                    or_helped_hq = p_helped_hq = ci_low_helped_hq = ci_high_helped_hq = None
                    or_ae_hq = p_ae_hq = ci_low_ae_hq = ci_high_ae_hq = None
                    test_method_helped_hq = test_method_ae_hq = "Insufficient data"
            else:
                # No HQ column
                n_hq_ref = n_hq_comp = None
                or_helped_hq = p_helped_hq = ci_low_helped_hq = ci_high_helped_hq = None
                or_ae_hq = p_ae_hq = ci_low_ae_hq = ci_high_ae_hq = None
                test_method_helped_hq = test_method_ae_hq = "N/A"
            
            # Format values for display with proper handling of None values
            def format_or(or_val, ci_low, ci_high, p_val):
                if or_val is None or p_val is None or ci_low is None or ci_high is None:
                    return "N/A"
                
                # Use more decimal places for very small values
                if 0 < or_val < 0.01 or 0 < ci_low < 0.01 or 0 < ci_high < 0.01:
                    or_text = f"{or_val:.3f} ({ci_low:.3f}-{ci_high:.3f})"
                else:
                    or_text = f"{or_val:.2f} ({ci_low:.2f}-{ci_high:.2f})"
                
                # Add significance stars
                if p_val < 0.001:
                    or_text += " ***"
                elif p_val < 0.01:
                    or_text += " **"
                elif p_val < 0.05:
                    or_text += " *"
                
                return or_text
            
            # Add record to ORs list
            all_ors.append({
                "Platform": platform,
                "Comparison": f"{ref_product} vs. {product}",
                "N_All_Ref": n_ref,
                "N_All_Comp": n_comp,
                "N_HQ_Ref": n_hq_ref,
                "HQ_Pct_Ref": round(100 * n_hq_ref / n_ref, 1) if n_hq_ref is not None and n_ref > 0 else None,
                "N_HQ_Comp": n_hq_comp,
                "HQ_Pct_Comp": round(100 * n_hq_comp / n_comp, 1) if n_hq_comp is not None and n_comp > 0 else None,
                "OR_Helped_All": format_or(or_helped, ci_low_helped, ci_high_helped, p_helped),
                "OR_Helped_HQ": format_or(or_helped_hq, ci_low_helped_hq, ci_high_helped_hq, p_helped_hq) 
                    if or_helped_hq is not None else "N/A",
                "OR_AE_All": format_or(or_ae, ci_low_ae, ci_high_ae, p_ae),
                "OR_AE_HQ": format_or(or_ae_hq, ci_low_ae_hq, ci_high_ae_hq, p_ae_hq) 
                    if or_ae_hq is not None else "N/A",
                "Test_Method_Helped_All": test_method_helped,
                "Test_Method_Helped_HQ": test_method_helped_hq,
                "Test_Method_AE_All": test_method_ae,
                "Test_Method_AE_HQ": test_method_ae_hq,
                # Raw values for sorting (with safe handling of None)
                "_or_helped": or_helped if or_helped is not None else 0,
                "_p_helped": p_helped if p_helped is not None else 1
            })
    
    # Convert to DataFrame and format
    if not all_ors:
        print(f"No odds ratio data available for the reference product: {ref_product}")
        return pd.DataFrame()
        
    or_df = pd.DataFrame(all_ors)
    
    # Sort by platform and significance
    or_df = or_df.sort_values(["Platform", "_p_helped"])
    
    # Drop helper columns used for sorting
    or_df = or_df.drop(columns=["_or_helped", "_p_helped"], errors="ignore")
    
    # Select columns for display
    display_cols = [
        "Platform", "Comparison", 
        "N_All_Ref", "N_All_Comp", 
        "OR_Helped_All", "Test_Method_Helped_All", 
        "OR_AE_All", "Test_Method_AE_All"
    ]
    
    # Add HQ columns if present
    if "N_HQ_Ref" in or_df.columns and not or_df["N_HQ_Ref"].isna().all():
        display_cols.extend([
            "N_HQ_Ref", "HQ_Pct_Ref", "N_HQ_Comp", "HQ_Pct_Comp",
            "OR_Helped_HQ", "Test_Method_Helped_HQ",
            "OR_AE_HQ", "Test_Method_AE_HQ"
        ])
    
    # Filter columns that exist
    display_cols = [col for col in display_cols if col in or_df.columns]
    
    return or_df[display_cols]

# or_table = create_odds_ratio_table(
#     ref_product="Chanca piedra", 
#     min_include_n=MIN_SAMPLE_SIZE,
#     min_regression_n=MIN_SAMPLE_SIZE,
#     min_events=MIN_EVENT_COUNT)
# display(or_table)
# or_table.to_csv("chanca_piedra_odds_ratio_table.csv", index=False)

In [ ]:
def create_odds_ratio_tables(ref_product="Chanca piedra", 
                           min_include_n=5,        # Minimum n to include in tables
                           min_regression_n=MIN_SAMPLE_SIZE,  # Threshold for regression vs Fisher
                           min_events=MIN_EVENT_COUNT):      # Minimum events for regression
    """
    Create four tables of odds ratios comparing the reference product (default: Chanca piedra)
    to other treatments across platforms:
    1. Helped - All Reviews
    2. Helped - High Quality
    3. Adverse Effects - All Reviews
    4. Adverse Effects - High Quality
    
    Parameters:
    -----------
    ref_product : str
        The reference product to compare other treatments against
    min_include_n : int
        Minimum number of reviews required for a product to be included in the analysis
    min_regression_n : int
        Minimum sample size to use logistic regression (vs Fisher's exact test)
    min_events : int
        Minimum number of events to use logistic regression (vs Fisher's exact test)
        
    Returns:
    --------
    tuple of pandas.DataFrame
        Four tables: helped all, helped HQ, AE all, AE HQ
    """
    # Call the original function to get the combined dataframe, passing all parameters
    exclude_from_helped = {"Garcinia", "Black seed", "Ashwagandha", "Melatonin"}
    or_df = create_odds_ratio_table(
        ref_product=ref_product, 
        min_include_n=min_include_n,
        min_regression_n=min_regression_n,
        min_events=min_events
    )
    
    if or_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    
    # Split the "Comparison" column into "Reference" and "Comparison"
    or_df["Reference"] = ref_product
    or_df["Comparator"] = or_df["Comparison"].apply(lambda x: x.split(" vs. ")[1] if " vs. " in x else "")
    
    # Create a copy for helped analysis that excludes certain medicines
    helped_df = or_df.copy()
    helped_df = helped_df[~((helped_df["Platform"] == "WebMD") & 
                        (helped_df["Comparator"].isin(exclude_from_helped)))]
    
    # Define column groups
    common_cols_all = ["Platform", "Reference", "Comparator", "N_All_Ref", "N_All_Comp"]
    common_cols_hq = ["Platform", "Reference", "Comparator", "N_HQ_Ref", "HQ_Pct_Ref", "N_HQ_Comp", "HQ_Pct_Comp"]
    
    helped_cols_all = ["OR_Helped_All", "Test_Method_Helped_All"]
    helped_cols_hq = ["OR_Helped_HQ", "Test_Method_Helped_HQ"]
    ae_cols_all = ["OR_AE_All", "Test_Method_AE_All"]
    ae_cols_hq = ["OR_AE_HQ", "Test_Method_AE_HQ"]
    
    # Filter to columns that exist in the dataframe
    common_cols_all = [col for col in common_cols_all if col in or_df.columns]
    common_cols_hq = [col for col in common_cols_hq if col in or_df.columns]
    helped_cols_all = [col for col in helped_cols_all if col in or_df.columns]
    helped_cols_hq = [col for col in helped_cols_hq if col in or_df.columns]
    ae_cols_all = [col for col in ae_cols_all if col in or_df.columns]
    ae_cols_hq = [col for col in ae_cols_hq if col in or_df.columns]
    
    # Create the four tables
    helped_table_all = helped_df[common_cols_all + helped_cols_all]
    helped_table_hq = helped_df[common_cols_hq + helped_cols_hq]
    ae_table_all = or_df[common_cols_all + ae_cols_all]
    ae_table_hq = or_df[common_cols_hq + ae_cols_hq]
    
    # Filter out rows where there's no HQ data for the HQ tables
    if "N_HQ_Ref" in helped_table_hq.columns:
        helped_table_hq = helped_table_hq[~helped_table_hq["N_HQ_Ref"].isna()]
    if "N_HQ_Ref" in ae_table_hq.columns:
        ae_table_hq = ae_table_hq[~ae_table_hq["N_HQ_Ref"].isna()]
    
    return helped_table_all, helped_table_hq, ae_table_all, ae_table_hq

# Call the updated function
# To include more medicines but still use Fisher for small samples:
helped_table_all, helped_table_hq, ae_table_all, ae_table_hq = create_odds_ratio_tables(
    ref_product="Chanca piedra", 
    min_include_n=MIN_EVENT_COUNT,  # Include medicines with at least 10 samples
    min_regression_n=MIN_SAMPLE_SIZE,  # Use regression only when n>=30
    min_events=MIN_EVENT_COUNT)  # Use regression only when events>=10

# Display all four tables
print("HELPED TABLE - ALL REVIEWS:")
display(helped_table_all)

print("\nHELPED TABLE - HIGH QUALITY:")
display(helped_table_hq)

print("\nADVERSE EFFECTS TABLE - ALL REVIEWS:")
display(ae_table_all)

print("\nADVERSE EFFECTS TABLE - HIGH QUALITY:")
display(ae_table_hq)

# Save all four tables to CSV
helped_table_all.to_csv(CSV_DIR / "chanca_piedra_helped_all_table.csv", index=False)
helped_table_hq.to_csv(CSV_DIR / "chanca_piedra_helped_hq_table.csv", index=False)
ae_table_all.to_csv(CSV_DIR / "chanca_piedra_ae_all_table.csv", index=False)
ae_table_hq.to_csv(CSV_DIR / "chanca_piedra_ae_hq_table.csv", index=False)

HELPED TABLE - ALL REVIEWS:


,Platform,Reference,Comparator,N_All_Ref,N_All_Comp,OR_Helped_All,Test_Method_Helped_All
8,Amazon,Chanca piedra,Potassium citrate,1193,133,3.84 (2.11-6.99) ***,Regression
9,Amazon,Chanca piedra,Rowatinex,1193,90,3.90 (1.88-8.11) ***,Regression
10,Amazon,Chanca piedra,Phosfood,1193,40,1.14 (0.56-2.35),Regression
12,Reddit,Chanca piedra,Potassium citrate,492,509,0.52 (0.39-0.69) ***,Regression
11,Reddit,Chanca piedra,Flomax,492,1134,0.60 (0.48-0.76) ***,Regression
13,Reddit,Chanca piedra,Allopurinol,492,49,0.47 (0.23-0.97) *,Regression
18,Reddit,Chanca piedra,Phosfood,492,2,9.21 (0.44-192.88),Fisher
16,Reddit,Chanca piedra,Rowatinex,492,21,0.58 (0.21-1.60),Fisher
15,Reddit,Chanca piedra,Garcinia,492,38,1.34 (0.69-2.62),Regression
17,Reddit,Chanca piedra,Black seed,492,17,0.57 (0.18-1.77),Fisher



HELPED TABLE - HIGH QUALITY:


,Platform,Reference,Comparator,N_HQ_Ref,HQ_Pct_Ref,N_HQ_Comp,HQ_Pct_Comp,OR_Helped_HQ,Test_Method_Helped_HQ
8,Amazon,Chanca piedra,Potassium citrate,158,13.2,26,19.5,1.72 (0.40-7.45),Fisher
9,Amazon,Chanca piedra,Rowatinex,158,13.2,20,22.2,5.92 (0.35-99.02),Fisher
10,Amazon,Chanca piedra,Phosfood,158,13.2,1,2.5,0.43 (0.02-10.74),Fisher
12,Reddit,Chanca piedra,Potassium citrate,75,15.2,147,28.9,0.42 (0.24-0.75) **,Regression
11,Reddit,Chanca piedra,Flomax,75,15.2,232,20.5,0.43 (0.25-0.73) **,Regression
13,Reddit,Chanca piedra,Allopurinol,75,15.2,8,16.3,0.63 (0.15-2.72),Fisher
18,Reddit,Chanca piedra,Phosfood,75,15.2,0,0.0,N/A,Insufficient data
16,Reddit,Chanca piedra,Rowatinex,75,15.2,3,14.3,4.44 (0.22-89.10),Fisher
15,Reddit,Chanca piedra,Garcinia,75,15.2,1,2.6,0.211 (0.008-5.366),Fisher
17,Reddit,Chanca piedra,Black seed,75,15.2,3,17.6,0.32 (0.03-3.63),Fisher



ADVERSE EFFECTS TABLE - ALL REVIEWS:


,Platform,Reference,Comparator,N_All_Ref,N_All_Comp,OR_AE_All,Test_Method_AE_All
8,Amazon,Chanca piedra,Potassium citrate,1193,133,0.85 (0.26-2.72),Fisher
9,Amazon,Chanca piedra,Rowatinex,1193,90,0.83 (0.20-3.45),Fisher
10,Amazon,Chanca piedra,Phosfood,1193,40,2.97 (0.90-9.86),Fisher
12,Reddit,Chanca piedra,Potassium citrate,492,509,2.22 (1.43-3.44) ***,Regression
11,Reddit,Chanca piedra,Flomax,492,1134,4.02 (2.74-5.91) ***,Regression
13,Reddit,Chanca piedra,Allopurinol,492,49,2.01 (0.79-5.06),Fisher
18,Reddit,Chanca piedra,Phosfood,492,2,2.83 (0.13-60.27),Fisher
16,Reddit,Chanca piedra,Rowatinex,492,21,2.40 (0.67-8.56),Fisher
15,Reddit,Chanca piedra,Garcinia,492,38,2.18 (0.80-5.96),Fisher
17,Reddit,Chanca piedra,Black seed,492,17,0.40 (0.02-6.88),Fisher



ADVERSE EFFECTS TABLE - HIGH QUALITY:


,Platform,Reference,Comparator,N_HQ_Ref,HQ_Pct_Ref,N_HQ_Comp,HQ_Pct_Comp,OR_AE_HQ,Test_Method_AE_HQ
8,Amazon,Chanca piedra,Potassium citrate,158,13.2,26,19.5,2.34 (0.28-19.18),Fisher
9,Amazon,Chanca piedra,Rowatinex,158,13.2,20,22.2,6.49 (1.31-32.25),Fisher
10,Amazon,Chanca piedra,Phosfood,158,13.2,1,2.5,18.47 (0.71-483.33),Fisher
12,Reddit,Chanca piedra,Potassium citrate,75,15.2,147,28.9,0.97 (0.44-2.13),Regression
11,Reddit,Chanca piedra,Flomax,75,15.2,232,20.5,1.94 (0.96-3.93),Regression
13,Reddit,Chanca piedra,Allopurinol,75,15.2,8,16.3,0.83 (0.09-7.43),Fisher
18,Reddit,Chanca piedra,Phosfood,75,15.2,0,0.0,N/A,Insufficient data
16,Reddit,Chanca piedra,Rowatinex,75,15.2,3,14.3,0.80 (0.04-16.57),Fisher
15,Reddit,Chanca piedra,Garcinia,75,15.2,1,2.6,16.83 (0.64-439.00),Fisher
17,Reddit,Chanca piedra,Black seed,75,15.2,3,17.6,0.80 (0.04-16.57),Fisher
